# Phishing & Typosquatting Hunt

This notebook is built for the **SCATTERED SPIDER phishing lab** and focuses on finding:
- look-alike domains
- typosquatting / masquerading infrastructure
- suspicious POST traffic consistent with credential harvesting
- rare or unusual domains and TLDs
- domains that do **not** appear in `../data/majestic_million.csv`

It loads `../data/lab_phishing.csv` and provides interactive hunting helpers for:
- filtering by **HTTP method**
- fuzzy matching against a **target brand/keyword**
- parsing **registered domains** and **TLDs**
- **LFO / thick-tail** analysis of domains and TLDs
- keyword search in the domain field
- highlighting and excluding domains **not** present in the Majestic Million
- additional phishing heuristics such as auth/login/SSO keywords, hyphenated look-alikes, and suspicious POST activity

> Tip: Run the notebook top to bottom first, then use the widgets and helper functions below to pivot quickly.

In [ ]:
# Imports
from pathlib import Path
from urllib.parse import urlparse, unquote
import re
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, HTML
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

try:
    import tldextract
except Exception as e:
    raise ImportError("tldextract is required for domain parsing. Install it with: pip install tldextract") from e

try:
    from rapidfuzz import fuzz
except Exception as e:
    raise ImportError("rapidfuzz is required for fuzzy matching. Install it with: pip install rapidfuzz") from e

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# --- Paths ---
DATA_PATH = Path("../data/lab_phishing.csv")
MAJESTIC_PATH = Path("../data/majestic_million.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find lab dataset: {DATA_PATH.resolve()}")

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded lab dataset: {DATA_PATH}")
print(f"Rows: {len(df_raw):,} | Columns: {len(df_raw.columns)}")
display(df_raw.head(3))

In [ ]:
# --- Column detection helpers ---
def find_first_matching_column(columns, candidates):
    cols_lower = {c.lower(): c for c in columns}
    for candidate in candidates:
        for c in columns:
            if c.lower() == candidate.lower():
                return c
    for candidate in candidates:
        for c in columns:
            if candidate.lower() in c.lower():
                return c
    return None

URL_CANDIDATES = [
    "URL", "Url", "Request URL", "URI", "URI Host", "Request", "full_url"
]
METHOD_CANDIDATES = [
    "HTTP Method", "Method", "Request Method", "Verb", "http_method"
]
TIME_CANDIDATES = [
    "Logged Time", "Event Time", "Time", "Timestamp"
]
ACTION_CANDIDATES = [
    "Policy Action", "Action", "Disposition"
]
USER_CANDIDATES = [
    "User", "Username", "Src User"
]
BYTES_CANDIDATES = [
    "Total Bytes", "Bytes", "Received Bytes"
]

url_col = find_first_matching_column(df_raw.columns, URL_CANDIDATES)
method_col = find_first_matching_column(df_raw.columns, METHOD_CANDIDATES)
time_col = find_first_matching_column(df_raw.columns, TIME_CANDIDATES)
action_col = find_first_matching_column(df_raw.columns, ACTION_CANDIDATES)
user_col = find_first_matching_column(df_raw.columns, USER_CANDIDATES)
bytes_col = find_first_matching_column(df_raw.columns, BYTES_CANDIDATES)

print("Detected columns")
print("----------------")
print("URL column:   ", url_col)
print("Method column:", method_col)
print("Time column:  ", time_col)
print("Action column:", action_col)
print("User column:  ", user_col)
print("Bytes column: ", bytes_col)

if not url_col:
    raise ValueError("Could not automatically detect the URL column. Update URL_CANDIDATES near the top of the notebook.")

In [ ]:
# --- URL/domain parsing ---
def clean_url(value):
    if pd.isna(value):
        return ""
    s = str(value).strip().strip('"').strip("'")
    s = unquote(s)
    return s

def ensure_scheme(url):
    # Some proxy datasets store only host/path, so add a scheme when needed for parsing
    if not url:
        return ""
    if "://" in url:
        return url
    return "https://" + url.lstrip("/")

def parse_url_parts(raw_url):
    raw = clean_url(raw_url)
    candidate = ensure_scheme(raw)
    parsed = urlparse(candidate)
    host = (parsed.netloc or "").lower()
    path = parsed.path or ""
    query = parsed.query or ""
    fragment = parsed.fragment or ""

    # Remove userinfo if present
    if "@" in host:
        host = host.split("@", 1)[-1]

    # Split port
    host_no_port = host.split(":", 1)[0]
    port = None
    if ":" in host:
        try:
            port = int(host.split(":", 1)[1])
        except Exception:
            port = None

    ext = tldextract.extract(host_no_port)
    subdomain = ext.subdomain.lower() if ext.subdomain else ""
    domain = ext.domain.lower() if ext.domain else ""
    suffix = ext.suffix.lower() if ext.suffix else ""
    registered_domain = ".".join(x for x in [domain, suffix] if x)
    fqdn = ".".join(x for x in [subdomain, registered_domain] if x)

    return {
        "url_clean": raw,
        "host": host,
        "host_no_port": host_no_port,
        "port": port,
        "path": path,
        "query": query,
        "fragment": fragment,
        "subdomain": subdomain,
        "domain": domain,
        "suffix": suffix,
        "registered_domain": registered_domain,
        "fqdn": fqdn,
        "path_depth": len([p for p in path.split("/") if p]),
        "host_label_count": len([p for p in host_no_port.split(".") if p]),
        "hyphen_count": host_no_port.count("-"),
        "digit_count": sum(ch.isdigit() for ch in host_no_port),
    }

parsed_df = df_raw[url_col].apply(parse_url_parts).apply(pd.Series)
df = pd.concat([df_raw.copy(), parsed_df], axis=1)

if method_col:
    df[method_col] = df[method_col].astype(str).str.upper().str.strip()

display(df[[c for c in [url_col, method_col, "host_no_port", "registered_domain", "suffix", "path"] if c in df.columns]].head(5))

In [ ]:
# --- Load Majestic Million and normalize domains ---
def load_majestic(path):
    if not path.exists():
        print(f"Majestic file not found at {path}. Features that rely on Majestic Million will be disabled.")
        return set(), None

    mm = pd.read_csv(path, low_memory=False)
    domain_guess = find_first_matching_column(mm.columns, ["Domain", "RootDomain", "domain", "rootdomain"])
    if not domain_guess:
        # Fallback: pick the first object/string-like column
        for c in mm.columns:
            if mm[c].dtype == "object":
                domain_guess = c
                break

    if not domain_guess:
        raise ValueError("Could not detect the domain column in majestic_million.csv")

    domains = (
        mm[domain_guess]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({"nan": np.nan})
        .dropna()
        .tolist()
    )
    domain_set = set(domains)
    return domain_set, mm

majestic_domains, majestic_df = load_majestic(MAJESTIC_PATH)

df["in_majestic_million"] = df["registered_domain"].isin(majestic_domains) if majestic_domains else False
df["not_in_majestic_million"] = ~df["in_majestic_million"]

print(f"Majestic loaded: {bool(majestic_domains)}")
if majestic_domains:
    print(f"Majestic domains available: {len(majestic_domains):,}")
    print(f"Rows in lab not in Majestic Million: {df['not_in_majestic_million'].sum():,}")

# Keywords
The following brand, authentication, and phishy words are defined in the cell below. Feel free to modify the lists. 


`BRAND_KEYWORDS = [
    "quickbooks", "appsheet", "canva", "surveymonkey", "google-docs", "googledocs", "zoom",
    "microsoft", "office365", "okta", "adobe", "docusign", "dropbox", "box", "slack"
]`

`AUTH_KEYWORDS = [
    "auth", "login", "logon", "signin", "sign-in", "sso", "identity", "account", "verify",
    "validate", "secure", "session", "password", "credential", "mfa", "2fa"
]`

`PHISHY_PATH_KEYWORDS = [
    "login", "signin", "auth", "sso", "password", "reset", "verify", "account", "mfa", "2fa"
]`

In [ ]:
# --- Threat hunting enrichment / phishing heuristics ---
BRAND_KEYWORDS = [
    "quickbooks", "appsheet", "canva", "surveymonkey", "google-docs", "googledocs", "zoom",
    "microsoft", "office365", "okta", "adobe", "docusign", "dropbox", "box", "slack"
]
AUTH_KEYWORDS = [
    "auth", "login", "logon", "signin", "sign-in", "sso", "identity", "account", "verify",
    "validate", "secure", "session", "password", "credential", "mfa", "2fa"
]
PHISHY_PATH_KEYWORDS = [
    "login", "signin", "auth", "sso", "password", "reset", "verify", "account", "mfa", "2fa"
]

def keyword_hits(text, keywords):
    text = (text or "").lower()
    return [kw for kw in keywords if kw.lower() in text]

df["domain_brand_hits"] = df["host_no_port"].apply(lambda x: keyword_hits(x, BRAND_KEYWORDS))
df["auth_keyword_hits"] = df["host_no_port"].apply(lambda x: keyword_hits(x, AUTH_KEYWORDS))
df["path_keyword_hits"] = df["path"].apply(lambda x: keyword_hits(x, PHISHY_PATH_KEYWORDS))
df["has_brand_keyword"] = df["domain_brand_hits"].apply(bool)
df["has_auth_keyword"] = df["auth_keyword_hits"].apply(bool)
df["has_phishy_path_keyword"] = df["path_keyword_hits"].apply(bool)
df["looks_like_auth_portal"] = df[["has_brand_keyword", "has_auth_keyword", "has_phishy_path_keyword"]].any(axis=1)

if method_col:
    df["is_post"] = df[method_col].eq("POST")
else:
    df["is_post"] = False

if action_col:
    df["action_lower"] = df[action_col].astype(str).str.lower()
    df["is_allowed"] = df["action_lower"].str.contains("allow|allowed|permit|permitted", regex=True, na=False)
else:
    df["is_allowed"] = False

df["suspicious_post_to_auth"] = df["is_post"] & df["looks_like_auth_portal"]
df["domain_length"] = df["host_no_port"].astype(str).str.len()
df["registered_domain_length"] = df["registered_domain"].astype(str).str.len()

risk_cols = ["suspicious_post_to_auth", "not_in_majestic_million", "has_brand_keyword", "has_auth_keyword", "has_phishy_path_keyword"]
df["heuristic_score"] = df[risk_cols].sum(axis=1)

display(df[[c for c in [url_col, method_col, action_col, "registered_domain", "in_majestic_million", "domain_brand_hits", "auth_keyword_hits", "path_keyword_hits", "heuristic_score"] if c in df.columns]].head(10))

## Quick summary pivots

The cells below create fast domain- and TLD-level pivots so you can:
- reduce noise to grouped domains
- inspect least-frequent observations (LFO / thin tail)
- inspect most-frequent observations (thick tail)
- compare POST-heavy and auth-looking infrastructure

In [ ]:
def summarize_by_domain(dataframe=None):
    d = df if dataframe is None else dataframe
    group_cols = {
        "events": ("registered_domain", "size"),
        "unique_fqdns": ("host_no_port", pd.Series.nunique),
        "post_events": ("is_post", "sum"),
        "allowed_events": ("is_allowed", "sum"),
        "not_in_majestic_events": ("not_in_majestic_million", "sum"),
        "auth_like_events": ("looks_like_auth_portal", "sum"),
        "suspicious_post_to_auth": ("suspicious_post_to_auth", "sum"),
        "max_heuristic_score": ("heuristic_score", "max"),
    }
    out = d.groupby("registered_domain").agg(**group_cols).reset_index()
    out["post_ratio"] = np.where(out["events"] > 0, out["post_events"] / out["events"], 0)
    out["auth_like_ratio"] = np.where(out["events"] > 0, out["auth_like_events"] / out["events"], 0)
    out["not_in_majestic_ratio"] = np.where(out["events"] > 0, out["not_in_majestic_events"] / out["events"], 0)
    return out.sort_values(["events", "suspicious_post_to_auth"], ascending=[False, False])

domain_summary = summarize_by_domain()
display(domain_summary.head(20))

print(f"Distinct registered domains: {df['registered_domain'].nunique():,}")
print(f"Distinct FQDNs/hosts:        {df['host_no_port'].nunique():,}")
print(f"Distinct TLDs:               {df['suffix'].nunique():,}")

In [ ]:
def summarize_by_tld(dataframe=None):
    d = df if dataframe is None else dataframe
    out = (
        d.groupby("suffix")
        .agg(
            events=("suffix", "size"),
            distinct_domains=("registered_domain", pd.Series.nunique),
            post_events=("is_post", "sum"),
            auth_like_events=("looks_like_auth_portal", "sum"),
            suspicious_post_to_auth=("suspicious_post_to_auth", "sum"),
            not_in_majestic_events=("not_in_majestic_million", "sum"),
        )
        .reset_index()
        .sort_values("events", ascending=False)
    )
    return out

tld_summary = summarize_by_tld()
display(tld_summary.head(20))

In [ ]:
def lfo_view(summary_df, key_col, count_col="events", n=25, ascending=True):
    # ascending=True = least frequent observed (thin tail / LFO)
    return summary_df.sort_values([count_col, key_col], ascending=[ascending, True]).head(n)

print("Least frequent observed domains (LFO / thin tail)")
display(lfo_view(domain_summary, "registered_domain", n=25, ascending=True))

print("Thick tail domains")
display(lfo_view(domain_summary, "registered_domain", n=25, ascending=False))

print("Least frequent observed TLDs")
display(lfo_view(tld_summary, "suffix", n=25, ascending=True))

print("Thick tail TLDs")
display(lfo_view(tld_summary, "suffix", n=25, ascending=False))

In [ ]:
def plot_top(summary_df, key_col, value_col="events", n=20, title=None):
    top = summary_df.sort_values(value_col, ascending=False).head(n).copy()
    plt.figure(figsize=(12, max(4, 0.35 * len(top))))
    plt.barh(top[key_col].astype(str), top[value_col])
    plt.gca().invert_yaxis()
    plt.xlabel(value_col)
    plt.ylabel(key_col)
    plt.title(title or f"Top {n} {key_col} by {value_col}")
    plt.tight_layout()
    plt.show()

plot_top(domain_summary, "registered_domain", "events", n=20, title="Top 20 registered domains by event count")
plot_top(tld_summary, "suffix", "events", n=15, title="Top 15 TLDs by event count")

## Fuzzy matching for look-alike domains

This section helps find domains similar to a legitimate brand such as `surveymonkey.com`,
while excluding the exact legitimate domain.

Examples:
- target keyword: `surveymonkey`
- exclude exact registered domain: `surveymonkey.com`
- minimum score: `80`



Modify this section to change the target keyword:
    
    fuzzy_results = fuzzy_match_domains(

    keyword="surveymonkey",
    
    exclude_exact="surveymonkey.com",
    
    min_score=75,
    
    only_not_in_majestic=False,
    
    require_hyphen_or_extra_label=False

In [ ]:
def fuzzy_match_domains(
    keyword,
    exclude_exact=None,
    min_score=80,
    only_not_in_majestic=False,
    require_hyphen_or_extra_label=False,
    dataframe=None,
):
    d = df if dataframe is None else dataframe
    candidates = d[["host_no_port", "registered_domain", "subdomain", "suffix", "in_majestic_million",
                    "not_in_majestic_million", "is_post", "looks_like_auth_portal", "heuristic_score"]].drop_duplicates().copy()

    keyword = str(keyword).strip().lower()
    exclude_exact = str(exclude_exact).strip().lower() if exclude_exact else None

    # Compare both host and registered domain
    candidates["score_registered_domain"] = candidates["registered_domain"].apply(lambda x: fuzz.ratio(keyword, str(x).lower()))
    candidates["score_host"] = candidates["host_no_port"].apply(lambda x: fuzz.partial_ratio(keyword, str(x).lower()))
    candidates["fuzzy_score"] = candidates[["score_registered_domain", "score_host"]].max(axis=1)

    if exclude_exact:
        candidates = candidates[candidates["registered_domain"].str.lower() != exclude_exact]
        candidates = candidates[candidates["host_no_port"].str.lower() != exclude_exact]

    if only_not_in_majestic:
        candidates = candidates[candidates["not_in_majestic_million"]]

    if require_hyphen_or_extra_label:
        candidates = candidates[
            candidates["host_no_port"].str.contains("-", regex=False) |
            (candidates["subdomain"].fillna("") != "")
        ]

    candidates = candidates[candidates["fuzzy_score"] >= min_score]
    candidates = candidates.sort_values(
        ["fuzzy_score", "heuristic_score", "looks_like_auth_portal", "is_post", "not_in_majestic_million", "host_no_port"],
        ascending=[False, False, False, False, False, True]
    )

    return candidates.reset_index(drop=True)

# Example hunt: SurveyMonkey look-alikes, excluding the legitimate domain
fuzzy_results = fuzzy_match_domains(
    keyword="surveymonkey",
    exclude_exact="surveymonkey.com",
    min_score=75,
    only_not_in_majestic=False,
    require_hyphen_or_extra_label=False
)
display(fuzzy_results.head(30))

## Keyword search in domains / URLs

Use this when you want to search for terms like:
- `survey`
- `auth`
- `login`
- `zoom`
- `quickbooks`
- `google-docs`

In [ ]:
def search_domain_keywords(
    keywords,
    search_url_too=False,
    method_filter=None,
    only_not_in_majestic=False,
    dataframe=None
):
    d = df if dataframe is None else dataframe
    if isinstance(keywords, str):
        keywords = [k.strip() for k in keywords.split("|") if k.strip()]
    keywords = [k.lower() for k in keywords]

    mask = pd.Series(False, index=d.index)
    for kw in keywords:
        mask = mask | d["host_no_port"].str.lower().str.contains(re.escape(kw), na=False)
        if search_url_too:
            mask = mask | d[url_col].astype(str).str.lower().str.contains(re.escape(kw), na=False)

    if method_filter and method_col:
        mask = mask & d[method_col].eq(str(method_filter).upper())

    if only_not_in_majestic:
        mask = mask & d["not_in_majestic_million"]

    cols = [c for c in [time_col, user_col, method_col, action_col, url_col, "host_no_port", "registered_domain", "suffix", "in_majestic_million", "looks_like_auth_portal", "heuristic_score"] if c in d.columns]
    return d.loc[mask, cols].sort_values(cols[0] if cols and cols[0] else url_col).reset_index(drop=True)

display(search_domain_keywords("surveymonkey|zoom|quickbooks", search_url_too=False, only_not_in_majestic=False).head(30))

## HTTP method filtering and POST-to-login hunting

Phishing and credential harvesting often appear as **allowed HTTPS POST** traffic to
look-alike infrastructure. The cells below make that easy to inspect.

In [ ]:
def filter_by_http_method(method="POST", dataframe=None):
    d = df if dataframe is None else dataframe
    if not method_col:
        raise ValueError("No HTTP method column was detected in this dataset.")
    return d[d[method_col].eq(str(method).upper())].copy()

post_df = filter_by_http_method("POST") if method_col else df.iloc[0:0].copy()
print(f"POST events: {len(post_df):,}")
display(post_df[[c for c in [time_col, method_col, action_col, url_col, "host_no_port", "registered_domain", "not_in_majestic_million", "looks_like_auth_portal", "heuristic_score"] if c in post_df.columns]].head(25))

In [ ]:
# POST-focused grouped domains
if len(post_df):
    post_domain_summary = summarize_by_domain(post_df).sort_values(
        ["suspicious_post_to_auth", "post_events", "events", "max_heuristic_score"],
        ascending=[False, False, False, False]
    )
    display(post_domain_summary.head(30))

    print("Least frequent POST domains (very useful for phishing hunts)")
    display(lfo_view(post_domain_summary, "registered_domain", n=30, ascending=True))

## Highlight or exclude domains not in Majestic Million

This is useful for surfacing less common infrastructure and removing well-known noise.

In [ ]:
def uncommon_domains_view(
    only_not_in_majestic=True,
    min_events=1,
    post_only=False,
    auth_like_only=False,
    dataframe=None
):
    d = df if dataframe is None else dataframe
    if post_only:
        d = d[d["is_post"]]
    if auth_like_only:
        d = d[d["looks_like_auth_portal"]]

    summary = summarize_by_domain(d)
    if only_not_in_majestic:
        summary = summary[summary["not_in_majestic_events"] > 0]
    summary = summary[summary["events"] >= min_events]

    return summary.sort_values(
        ["suspicious_post_to_auth", "post_ratio", "auth_like_ratio", "events"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

display(uncommon_domains_view(only_not_in_majestic=True, min_events=1, post_only=False, auth_like_only=False).head(50))

## Additional phishing helpers

These functions add quick pivots commonly useful in phishing / credential-harvesting hunts:
1. domains with auth/login/SSO words
2. hyphenated domains containing a brand
3. domains with suspicious POST activity
4. investigation view for a single domain

In [ ]:
def auth_like_domain_summary(dataframe=None):
    d = df if dataframe is None else dataframe
    s = summarize_by_domain(d[d["looks_like_auth_portal"]])
    return s.sort_values(
        ["suspicious_post_to_auth", "post_events", "auth_like_events", "events"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

display(auth_like_domain_summary().head(50))

In [ ]:
def hyphenated_brand_domains(brands=None, dataframe=None):
    d = df if dataframe is None else dataframe
    brands = brands or BRAND_KEYWORDS
    tmp = d[["host_no_port", "registered_domain", "subdomain", "suffix", "is_post", "not_in_majestic_million", "heuristic_score"]].drop_duplicates().copy()
    tmp = tmp[tmp["host_no_port"].str.contains("-", regex=False)]
    pattern = "|".join(re.escape(b) for b in brands)
    tmp = tmp[tmp["host_no_port"].str.contains(pattern, case=False, regex=True, na=False)]
    return tmp.sort_values(
        ["heuristic_score", "not_in_majestic_million", "is_post", "host_no_port"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

display(hyphenated_brand_domains(["surveymonkey", "zoom", "quickbooks", "google-docs", "canva"]).head(50))

In [ ]:
def investigate_domain(domain, dataframe=None):
    d = df if dataframe is None else dataframe
    q = str(domain).strip().lower()
    mask = (
        d["host_no_port"].str.lower().eq(q) |
        d["registered_domain"].str.lower().eq(q)
    )
    out = d.loc[mask].copy()
    cols = [c for c in [time_col, user_col, method_col, action_col, url_col, "host", "host_no_port", "registered_domain", "suffix", "path", "query", "in_majestic_million", "domain_brand_hits", "auth_keyword_hits", "path_keyword_hits", "heuristic_score"] if c in out.columns]
    return out[cols].sort_values(time_col if time_col in cols else cols[0]).reset_index(drop=True)

# Example:
# display(investigate_domain("auth-us-surveymonkey.com"))

## Interactive hunting widgets

If `ipywidgets` is available in your environment, the next cell provides an interactive console
for filtering by:
- HTTP method
- a domain keyword
- a fuzzy-match target
- exclusion of the legitimate domain
- only showing domains not in Majestic Million
- minimum fuzzy score

If widgets are not available, skip this section and use the helper functions above.

In [ ]:
if WIDGETS_AVAILABLE:
    method_options = ["ALL"] + (sorted(df[method_col].dropna().astype(str).str.upper().unique().tolist()) if method_col else ["ALL"])
    method_dd = widgets.Dropdown(options=method_options, value="ALL", description="Method:")
    keyword_box = widgets.Text(value="", description="Keyword:")
    fuzzy_target_box = widgets.Text(value="surveymonkey", description="Fuzzy target:")
    exclude_exact_box = widgets.Text(value="surveymonkey.com", description="Exclude:")
    min_score_slider = widgets.IntSlider(value=75, min=50, max=100, step=1, description="Min score:")
    only_not_mm = widgets.Checkbox(value=False, description="Only not in Majestic")
    auth_only = widgets.Checkbox(value=False, description="Auth-like only")
    post_only = widgets.Checkbox(value=False, description="POST only")
    run_button = widgets.Button(description="Run Hunt", button_style="primary")
    output = widgets.Output()

    def run_hunt(_=None):
        with output:
            output.clear_output()
            d = df.copy()

            if method_col and method_dd.value != "ALL":
                d = d[d[method_col].eq(method_dd.value)]
            if auth_only.value:
                d = d[d["looks_like_auth_portal"]]
            if post_only.value:
                d = d[d["is_post"]]

            if keyword_box.value.strip():
                results = search_domain_keywords(
                    keyword_box.value.strip(),
                    search_url_too=False,
                    method_filter=None,
                    only_not_in_majestic=only_not_mm.value,
                    dataframe=d
                )
                print("Keyword search results")
                display(results.head(100))

            if fuzzy_target_box.value.strip():
                fuzzy = fuzzy_match_domains(
                    keyword=fuzzy_target_box.value.strip(),
                    exclude_exact=exclude_exact_box.value.strip() or None,
                    min_score=min_score_slider.value,
                    only_not_in_majestic=only_not_mm.value,
                    require_hyphen_or_extra_label=False,
                    dataframe=d
                )
                print("Fuzzy match candidates")
                display(fuzzy.head(100))

            print("Grouped domain summary")
            summary = summarize_by_domain(d)
            if only_not_mm.value:
                summary = summary[summary["not_in_majestic_events"] > 0]
            summary = summary.sort_values(
                ["suspicious_post_to_auth", "post_events", "auth_like_events", "events"],
                ascending=[False, False, False, False]
            )
            display(summary.head(100))

    run_button.on_click(run_hunt)
    display(widgets.VBox([
        widgets.HBox([method_dd, min_score_slider]),
        widgets.HBox([keyword_box, fuzzy_target_box]),
        widgets.HBox([exclude_exact_box]),
        widgets.HBox([only_not_mm, auth_only, post_only]),
        run_button,
        output
    ]))
else:
    print("ipywidgets is not available in this environment. Use the helper functions above instead.")

## Suggested hunt workflow for this lab

A good workflow is:
1. Group by `registered_domain` to reduce noise.
2. Pivot on **POST** requests.
3. Search for targeted SaaS brands like:
   - `surveymonkey`
   - `zoom`
   - `quickbooks`
   - `google-docs`
   - `canva`
   - `appsheet`
4. Exclude the exact legitimate domain when needed.
5. Use fuzzy matching to find look-alikes.
6. Prioritize domains that are:
   - **not** in Majestic Million
   - contain **auth/login/SSO** terms
   - receive **POST** requests
   - have low frequency but strong phishing characteristics

In [ ]:
# Example hunting sequence for this lab
brands_of_interest = ["quickbooks", "appsheet", "canva", "surveymonkey", "google-docs", "zoom"]

print("1) Keyword pivot on brands")
display(search_domain_keywords("|".join(brands_of_interest), search_url_too=False).head(50))

print("\n2) SurveyMonkey fuzzy matches excluding the legitimate domain")
display(
    fuzzy_match_domains(
        keyword="surveymonkey",
        exclude_exact="surveymonkey.com",
        min_score=75,
        only_not_in_majestic=False
    ).head(30)
)

print("\n3) POST + auth-like + uncommon domains")
display(
    uncommon_domains_view(
        only_not_in_majestic=True,
        min_events=1,
        post_only=True,
        auth_like_only=True
    ).head(30)
)